# 02 - Custom Experiment

Register a new strategy and run explore (and optionally commit).

Each experiment is a small module under `quant_lib/experiments/` that builds a `Hypothesis` + config and calls `register()`.

## 1. Define your hypothesis

In [ ]:
from quant_lib.audit import for_vol_compression
from quant_lib.experiments import (
    PeriodConfig, UniverseConfig, StrategyConfig,
    ExperimentConfig, register,
)

# Write narrative fields before peeking at evaluation data.
# Factories set strategy_type + default search space.
hyp = for_vol_compression(
    name="my_breakout_v1",
    mechanism=(
        "Volatility compression (vol_pct_rank < 0.15) "
        "followed by volume-confirmed breakout of the 20-bar "
        "high generates intraday momentum in liquid crypto "
        "perpetuals."
    ),
    boundary_conditions=(
        "Fails in low-vol accumulation regimes where no "
        "breakout follows compression. Fails on illiquid "
        "pairs (< 50M USD daily volume)."
    ),
    success_criteria="SPA p < 0.15, PF > 1.3, min 30 OOS trades",
    entry_logic="vol_pct_rank < 0.15 + close > HH_20 + rvol > 3.0",
    exit_logic="Trailing stop at ATR x 3.0, bailout at 36 bars",
)

## 2. Build the config

In [ ]:
config = ExperimentConfig(
    name="my_breakout_v1",
    strategy_type="vol_compression",  # matches Hypothesis
    hypothesis=hyp,
    period=PeriodConfig(
        train_start="2021-07-01",
        train_end="2025-12-31",
        # holdout auto-resolves post-training (default 6 months).
    ),
    universe=UniverseConfig(
        symbols=["BTCUSDT", "ETHUSDT", "SOLUSDT"],
        min_volume_usdt=50_000_000,  # trailing 30d USD volume
        min_age_days=365,
    ),
    strategy=StrategyConfig(
        leverage=3.0,
        pf_weight_clamp_floor=0.5,
        pf_weight_clamp_ceiling=1.5,
    ),
)

## 3. Register and run explore

In [ ]:
import quant_lib

register(config)
result = quant_lib.run_explore(
    experiment_name="my_breakout_v1",
    cache_dir="./data_cache",
)
print(result.spa_p_value, result.n_oos_trades)

## 4. Optional holdout commit

Explore returns SPA/trades only. Holdout PSR is on commit:

```python
if result.spa_p_value < 0.05:
    commit_result = quant_lib.run_commit(
        experiment_name="my_breakout_v1",
        cache_dir="./data_cache",
    )
    print(f"Holdout PSR: {commit_result.psr:.4f}")
    print(f"Final equity: ${commit_result.final_equity:,.2f}")
```

**Warning:** `run_commit` is irreversible for that seal. Paper-grade `scripts/reproduce.py` does not call it.